# Data loss detection (DLP)

# Coding problem, 5.6.b: DLP egress rule

**Skill:** Detect Attacks · **Maps to:** 5.6.B · **Language:** Python 3 (pytest)

## Problem
Implement the volume/behavior-based DLP rule described in the lesson: given a list of outbound
transfer events, flag the ones that exceed a size threshold AND go to a destination that is not
on the organization's approved-destination list, a classic egress-monitoring rule.

## Function contract
Implement `solve(transfers: list, approved_destinations: list, volume_threshold_mb: int) -> list`
in the code editor (see the Reference section below).

## Input
- `transfers`: `list[dict]`, each `{"id": str, "bytes_mb": int, "destination": str}`.
- `approved_destinations`: `list[str]` of known-good destination names.
- `volume_threshold_mb`: `int`, transfer must exceed this size to be eligible for flagging.

## Output
Sorted `list[str]` of flagged transfer `id`s.

## Examples
```
transfers = [{"id":"t1","bytes_mb":500,"destination":"unknown-cloud.example"}]
solve(transfers, ["corp-backup.internal"], volume_threshold_mb=100) -> ["t1"]
```

```
transfers = [{"id":"t2","bytes_mb":500,"destination":"corp-backup.internal"}]
solve(transfers, ["corp-backup.internal"], volume_threshold_mb=100) -> []   # approved destination
```

## Constraints
- Deterministic, offline, standard library only.
- The size comparison is strict (`bytes_mb > volume_threshold_mb`).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(transfers: list, approved_destinations: list, volume_threshold_mb: int) -> list
```

### Input
- `transfers`: `list[dict]`, each `{"id": str, "bytes_mb": int, "destination": str}`. `id` is
  unique per transfer.
- `approved_destinations`: `list[str]` of destination names considered safe/sanctioned.
- `volume_threshold_mb`: `int`, a transfer must exceed this size (strictly greater than) to be
  eligible for flagging.

### Output (`result`)
Sorted `list[str]` of transfer `id`s that should be flagged: `bytes_mb > volume_threshold_mb`
AND `destination` is NOT in `approved_destinations`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- The size comparison is strict (`>`, not `>=`): a transfer exactly at the threshold is not
  flagged.
- An empty `approved_destinations` list means every destination is treated as unapproved.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 5.6.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict, List


def solve(
    transfers: List[Dict[str, object]],
    approved_destinations: List[str],
    volume_threshold_mb: int,
) -> List[str]:
    """DLP egress rule: flag outbound transfers that exceed a volume threshold AND go to a
    destination that is not on the approved list.

    Args:
        transfers: list of {"id": str, "bytes_mb": int, "destination": str}.
        approved_destinations: list[str] of known-good destination names.
        volume_threshold_mb: int — a transfer must exceed this size (strictly greater than)
            to be eligible for flagging.
    Returns:
        sorted list[str] of the transfer `id`s that should be flagged.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_large_transfer_to_unapproved_flagged():
    transfers = [{"id": "t1", "bytes_mb": 500, "destination": "unknown-cloud.example"}]
    assert impl.solve(transfers, ["corp-backup.internal"], volume_threshold_mb=100) == ["t1"]


def test_large_transfer_to_approved_not_flagged():
    transfers = [{"id": "t2", "bytes_mb": 500, "destination": "corp-backup.internal"}]
    assert impl.solve(transfers, ["corp-backup.internal"], volume_threshold_mb=100) == []


def test_small_transfer_to_unapproved_not_flagged():
    transfers = [{"id": "t3", "bytes_mb": 5, "destination": "unknown-cloud.example"}]
    assert impl.solve(transfers, ["corp-backup.internal"], volume_threshold_mb=100) == []


def test_multiple_transfers_only_matching_flagged():
    transfers = [
        {"id": "a", "bytes_mb": 200, "destination": "evil.example"},
        {"id": "b", "bytes_mb": 50, "destination": "evil.example"},
        {"id": "c", "bytes_mb": 300, "destination": "partner.trusted"},
        {"id": "d", "bytes_mb": 400, "destination": "another-evil.example"},
    ]
    result = impl.solve(transfers, ["partner.trusted"], volume_threshold_mb=100)
    assert result == ["a", "d"]


def test_results_sorted():
    transfers = [
        {"id": "zeta", "bytes_mb": 200, "destination": "bad.example"},
        {"id": "alpha", "bytes_mb": 200, "destination": "bad.example"},
    ]
    assert impl.solve(transfers, [], volume_threshold_mb=100) == ["alpha", "zeta"]


# ---- hidden / edge cases -------------------------------------------------
def test_boundary_exactly_at_threshold_not_flagged():
    transfers = [{"id": "t4", "bytes_mb": 100, "destination": "unapproved.example"}]
    assert impl.solve(transfers, [], volume_threshold_mb=100) == []


def test_boundary_one_over_threshold_flagged():
    transfers = [{"id": "t5", "bytes_mb": 101, "destination": "unapproved.example"}]
    assert impl.solve(transfers, [], volume_threshold_mb=100) == ["t5"]


def test_empty_approved_list_means_everything_unapproved():
    transfers = [{"id": "t6", "bytes_mb": 999, "destination": "anywhere.example"}]
    assert impl.solve(transfers, [], volume_threshold_mb=100) == ["t6"]


def test_empty_transfers():
    assert impl.solve([], ["ok.example"], volume_threshold_mb=100) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
